In [11]:
import pandas as pd
import openpyxl

pd.set_option('display.max_columns', None)


# Incidents Event Logs -> eigenlijk niet geschikt!
https://archive.ics.uci.edu/dataset/498/incident+management+process+enriched+event+log

* we have a SUPPORT GROUP (exercise level) -> maar een nummer kan dus ook meerdere groepen hebben...
* and each group has multiple CASES (session level)
* and each case has multiple UPDATES (interaction level or event level, one lower than CASES here)

Each case is solved, recommended within SLA time (An SLA (service level agreement) is an agreement between the provider and client that outlines measurable metrics). We know whether this is made or not.

We could find out if we can identify CASES where they don't make the SLA in time compared to others with similar problems thus in similar GROUPS based on information about the UPDATES

We have sequences (incident states, updates, update people) and itemsets (e.g. helpers ).


Downside: fewer levels than ours, but still, they are levels and a good use case.

In [12]:
df = pd.read_csv("Data/Public/accidents/incident_event_log.csv")
df.head()

,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,sys_created_by,sys_created_at,sys_updated_by,sys_updated_at,contact_type,location,category,subcategory,u_symptom,cmdb_ci,impact,urgency,priority,assignment_group,assigned_to,knowledge,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
0,INC0000045,New,True,0,0,0,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 21,29/2/2016 01:23,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
1,INC0000045,Resolved,True,0,0,2,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 642,29/2/2016 08:53,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
2,INC0000045,Resolved,True,0,0,3,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 804,29/2/2016 11:29,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
3,INC0000045,Closed,False,0,0,4,True,Caller 2403,Opened by 8,29/2/2016 01:16,Created by 6,29/2/2016 01:23,Updated by 908,5/3/2016 12:00,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,?,2 - Medium,2 - Medium,3 - Moderate,Group 56,?,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 149,29/2/2016 11:29,5/3/2016 12:00
4,INC0000047,New,True,0,0,0,True,Caller 2403,Opened by 397,29/2/2016 04:40,Created by 171,29/2/2016 04:57,Updated by 746,29/2/2016 04:57,Phone,Location 165,Category 40,Subcategory 215,Symptom 471,?,2 - Medium,2 - Medium,3 - Moderate,Group 70,Resolver 89,True,False,Do Not Notify,?,?,?,?,code 5,Resolved by 81,1/3/2016 09:52,6/3/2016 10:00


In [13]:
df.sort_values('assignment_group')

,number,incident_state,active,reassignment_count,reopen_count,sys_mod_count,made_sla,caller_id,opened_by,opened_at,sys_created_by,sys_created_at,sys_updated_by,sys_updated_at,contact_type,location,category,subcategory,u_symptom,cmdb_ci,impact,urgency,priority,assignment_group,assigned_to,knowledge,u_priority_confirmation,notify,problem_id,rfc,vendor,caused_by,closed_code,resolved_by,resolved_at,closed_at
13371,INC0003089,Awaiting User Info,True,3,0,16,True,Caller 439,Opened by 131,7/3/2016 08:47,?,?,Updated by 44,22/3/2016 12:34,Phone,Location 143,Category 55,Subcategory 170,Symptom 607,?,3 - Low,3 - Low,4 - Low,?,Resolver 13,True,False,Do Not Notify,?,?,?,?,code 1,Resolved by 133,22/3/2016 13:48,27/3/2016 13:59
94173,INC0022723,New,True,1,0,1,True,Caller 2293,Opened by 40,26/4/2016 11:18,Created by 20,26/4/2016 11:19,Updated by 88,26/4/2016 11:19,Phone,Location 56,Category 26,Subcategory 275,?,?,2 - Medium,2 - Medium,3 - Moderate,?,Resolver 33,False,False,Do Not Notify,?,?,?,?,code 7,Resolved by 32,2/5/2016 12:15,7/5/2016 13:07
28929,INC0006526,Active,True,1,0,2,True,Caller 4760,Opened by 397,14/3/2016 10:08,Created by 171,14/3/2016 10:11,Updated by 573,14/3/2016 12:08,Phone,Location 143,Category 46,Subcategory 223,Symptom 491,?,2 - Medium,2 - Medium,3 - Moderate,?,Resolver 17,False,False,Do Not Notify,?,?,?,?,code 7,Resolved by 15,14/3/2016 13:26,24/3/2016 18:52
41387,INC0009306,Active,True,1,0,1,True,Caller 2449,Opened by 17,21/3/2016 09:56,Created by 10,21/3/2016 10:07,Updated by 44,21/3/2016 19:41,Phone,Location 161,Category 26,Subcategory 174,Symptom 491,?,2 - Medium,2 - Medium,3 - Moderate,?,Resolver 253,False,False,Do Not Notify,?,?,?,?,code 7,Resolved by 231,25/3/2016 13:52,30/3/2016 13:59
33239,INC0007406,New,True,1,0,1,True,Caller 2374,Opened by 55,15/3/2016 16:04,Created by 30,15/3/2016 16:40,Updated by 129,15/3/2016 16:40,Phone,Location 204,Category 42,Subcategory 223,?,?,2 - Medium,2 - Medium,3 - Moderate,?,Resolver 100,False,False,Do Not Notify,?,?,?,?,code 6,Resolved by 90,30/8/2016 11:20,4/9/2016 12:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64762,INC0015098,Resolved,True,2,0,74,True,Caller 2251,Opened by 17,4/4/2016 17:48,Created by 10,4/4/2016 17:49,Updated by 241,31/10/2016 15:57,Phone,Location 108,Category 34,Subcategory 134,Symptom 465,?,2 - Medium,2 - Medium,3 - Moderate,Group 9,Resolver 34,False,True,Do Not Notify,?,?,?,?,code 6,Resolved by 62,31/10/2016 15:57,5/11/2016 16:00
64763,INC0015098,Closed,False,2,0,75,False,Caller 2251,Opened by 17,4/4/2016 17:48,Created by 10,4/4/2016 17:49,Updated by 908,5/11/2016 16:00,Phone,Location 108,Category 34,Subcategory 134,Symptom 465,?,2 - Medium,2 - Medium,3 - Moderate,Group 9,Resolver 34,False,True,Do Not Notify,?,?,?,?,code 6,Resolved by 62,31/10/2016 15:57,5/11/2016 16:00
22972,INC0005074,New,True,2,0,4,True,Caller 3737,Opened by 108,9/3/2016 17:01,Created by 52,9/3/2016 17:14,Updated by 507,11/3/2016 13:01,Phone,Location 93,Category 34,Subcategory 134,Symptom 465,?,2 - Medium,2 - Medium,3 - Moderate,Group 9,Resolver 215,True,False,Do Not Notify,?,?,?,?,code 6,Resolved by 196,5/4/2016 09:21,10/4/2016 09:59
64758,INC0015098,New,True,0,0,20,True,Caller 2251,Opened by 17,4/4/2016 17:48,Created by 10,4/4/2016 17:49,Updated by 241,1/6/2016 16:47,Phone,Location 108,Category 34,Subcategory 134,Symptom 465,?,2 - Medium,2 - Medium,3 - Moderate,Group 9,Resolver 34,False,False,Do Not Notify,?,?,?,?,code 6,Resolved by 62,31/10/2016 15:57,5/11/2016 16:00


In [14]:
# levels: support group -> number -> ...
# c2, support group = 

descriptors = []


targets = []

ids = ['number']

In [16]:
for nr in list(df['number'].unique()):
    df_nr = df[ df['number'] == nr ]
    print(df_nr['problem_id'].unique())

['?']
['?']
['?' 'Problem ID  2']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['Problem ID  2']
['?']
['?']
['?']
['?']
['?']
['Problem ID  2']
['?']
['Problem ID  2']
['Problem ID  2']
['Problem ID  2']
['?']
['?']
['?' 'Problem ID  2']
['?']
['?']
['Problem ID  2']
['?']
['Problem ID  2']
['?']
['?']
['Problem ID  2']
['?']
['?']
['Problem ID  2']
['Problem ID  2']
['?']
['?']
['?' 'Problem ID  2']
['?']
['?']
['?']
['?']
['?']
['Problem ID  2']
['?']
['Problem ID  2']
['Problem ID  2']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['Problem ID  2']
['Problem ID  2']
['Problem ID  2']
['Problem ID  2']
['Problem ID  2']
['?']
['Problem ID  2']
['Problem ID  2']
['Problem ID  2']
['?']
['?']
['?']
['?']
['Problem ID  2']
['?']
['?']
['?']
['Problem ID  2']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['Problem ID  2']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?']
['?'

KeyboardInterrupt: 